# Ingestão SIH/SUS e agregação — Diabetes no SUS

Este notebook baixa os microdados do SIH/SUS (Sistema de Informações Hospitalares),
filtra as internações por diabetes (CID-10 E10-E14), aplica a padronização de idade
e faixa etária do projeto, agrega tudo até a camada `silver` com PySpark e, por fim,
junta com população e cobertura de Atenção Primária à Saúde (APS) para gerar a
camada `gold` (`municipio_ano.csv`), usada nas análises de desigualdade regional no
cuidado ao diabetes.

**Este notebook roda no Google Colab, não localmente.** Os microdados do DATASUS são
distribuídos em formato `.dbc` (DBF compactado), que exige a biblioteca
`datasus-dbc` — sem *wheel* disponível para Python 3.13 no Windows, o que já foi
verificado e está documentado na Seção 2.4 do spec do projeto. Por isso todo o
processamento de ingestão e a agregação Spark acontecem no ambiente do Colab
(Linux, Python compatível), e apenas o resultado final (`municipio_ano.csv`) volta
para a máquina local.

**Janela de dados:** 2019 a 2024, as 27 UFs, 1.944 arquivos mensais esperados
(27 UFs × 6 anos × 12 meses).

**Regras de recorte:**
- Diagnóstico principal (`DIAG_PRINC`) começando em E10, E11, E12, E13 ou E14.
- Amputação de membro inferior: procedimento SIGTAP com prefixo `040805`.
- AIHs de continuação (`IDENT == 5`) são excluídas — não são novas internações.

## Parte 1 — Camada bronze (ingestão)

As células a seguir baixam cada arquivo mensal do FTP do DATASUS, descompactam o
`.dbc`, filtram as internações de diabetes e gravam um parquet particionado por UF
e ano em `bronze/uf=<UF>/ano=<AAAA>/RD<UF><AAMM>.parquet` no Google Drive.

### 1.1 Configuracao do ambiente

Monta o Google Drive, instala as dependencias e **clona o repositorio publico**.

O Drive serve para o que precisa sobreviver a uma queda de sessao: as camadas
`bronze/` e `silver/`, que custam horas de download. Ja o **codigo e os insumos
vem do repositorio**, clonado a cada sessao — nao ha upload manual de nada, e o
que roda aqui e exatamente o que esta versionado no GitHub.

Dependencias instaladas:

- `datasus-dbc`: descompacta o formato `.dbc` do DATASUS para `.dbf`.
- `dbfread`: le o `.dbf` resultante como um DataFrame do pandas.
- `pyarrow`: grava os arquivos intermediarios em parquet.


In [ ]:
!pip install -q datasus-dbc pyarrow dbfread
from google.colab import drive
drive.mount('/content/drive')

import os

# Drive: so o que precisa sobreviver a queda de sessao.
BASE = '/content/drive/MyDrive/diabetes_sus'
for pasta in ('bronze', 'silver', 'gold', 'logs'):
    os.makedirs(f'{BASE}/{pasta}', exist_ok=True)

# Repositorio publico: codigo + insumos, sempre na versao versionada.
URL = 'https://github.com/Maykongc/projeto-diabetes-sus.git'
REPO = '/content/projeto-diabetes-sus'
if os.path.exists(REPO):
    os.system(f'git -C {REPO} pull -q')
else:
    os.system(f'git clone --depth 1 -q {URL} {REPO}')

assert os.path.exists(f'{REPO}/src/diabetes_sus/__init__.py'), 'clone falhou'
print('BASE (Drive):', BASE)
print('REPO (clone):', REPO)


### 1.2 Modulos do projeto

As funcoes de filtro, conversao de idade e compatibilizacao de codigos de
municipio vem do pacote `src/diabetes_sus/` do repositorio — o mesmo pacote
coberto por 85 testes localmente. Elas nao sao reimplementadas aqui, para que
nao exista divergencia entre o que e testado e o que roda no Colab.

Como o repositorio ja foi clonado na celula anterior, **nao ha nenhum passo
manual**: basta apontar o `sys.path` para dentro do clone.


In [ ]:
import sys
sys.path.insert(0, f'{REPO}/src')
from diabetes_sus.filtros import filtrar_internacoes_diabetes
from diabetes_sus.idade import faixa_etaria, idade_em_anos
print('modulos carregados do clone')


### 1.3 Catalogo de arquivos a baixar

O SIH/SUS publica um arquivo `.dbc` por UF, ano e mes. Em vez de montar o produto
cartesiano 27 x 6 x 12 e descobrir por tentativa e erro o que existe, esta celula
**lista o diretorio no servidor** e monta o catalogo a partir do que realmente esta
la, com o tamanho de cada arquivo. Um arquivo ausente vira ausencia conhecida, nao
tres tentativas de download seguidas de uma linha de FALHOU.

A listagem tambem da o volume total, que dimensiona a espera: sao cerca de
**5,2 GB** e **1.944 arquivos**.

**Transporte: `ftp://`, nao `https://`.** O host `ftp.datasus.gov.br` nao tem
servidor web — as portas 80 e 443 recusam conexao, so a 21 responde. Uma versao
anterior deste notebook usava `https://` e falhava em 100% dos downloads com
`Connection timed out`.


In [ ]:
import re
from ftplib import FTP

ANOS = range(2019, 2025)
HOST = 'ftp.datasus.gov.br'
DIR_FTP = '/dissemin/publicos/SIHSUS/200801_/Dados'

# O catalogo vem de uma listagem real do servidor, nao de um produto cartesiano
# de UF x ano x mes. Assim, arquivo que nao existe aparece aqui como ausencia
# conhecida, em vez de virar 3 tentativas de download e uma linha de FALHOU.
con = FTP(HOST, timeout=180)
con.login()
con.cwd(DIR_FTP)
linhas = []
con.retrlines('LIST RD*.dbc', linhas.append)
con.quit()

# Formato do servidor (estilo IIS): '03-10-20  02:42PM   237472 RDAC1901.dbc'
padrao = re.compile(r'\s(\d+)\s+RD(\w{2})(\d{2})(\d{2})\.dbc\s*$', re.IGNORECASE)

catalogo = {}
for linha in linhas:
    achado = padrao.search(linha)
    if not achado:
        continue
    tamanho = int(achado.group(1))
    uf = achado.group(2).upper()
    ano = 2000 + int(achado.group(3))
    mes = int(achado.group(4))
    if ano in ANOS and 1 <= mes <= 12:
        catalogo[(uf, ano, mes)] = tamanho

alvos = sorted(catalogo)
volume = sum(catalogo.values())

print(f'{len(alvos)} arquivos disponiveis no servidor (27 UFs x 12 meses x 6 anos = 1944)')
print(f'volume total: {volume / 1024 ** 3:.2f} GB')
if len(alvos) != 1944:
    faltando = 1944 - len(alvos)
    print(f'AVISO: {faltando} combinacao(oes) UF/ano/mes nao existem no servidor')


### 1.4 Ingestao paralela com checkpoint

Para cada arquivo do catalogo: baixa o `.dbc` por FTP, descompacta para `.dbf`, le
com `dbfread`, aplica `filtrar_internacoes_diabetes` (que ja exclui AIHs de
continuacao e marca amputacao de MMII), converte idade e faixa etaria, e grava um
parquet por arquivo de origem.

**Seis conexoes em paralelo.** Medido contra o servidor real: uma conexao unica
sustenta 0,50 MB/s, seis sustentam 2,39 MB/s — 4,8x. Para os 5,2 GB isso e a
diferenca entre cerca de tres horas e cerca de 40 minutos. Cada worker mantem sua
propria conexao FTP aberta durante toda a sua fatia, em vez de reabrir uma por
arquivo.

**Checkpoint por arquivo.** Antes de baixar, verifica se o parquet de destino ja
existe e, se sim, pula. Da para reexecutar a celula quantas vezes for preciso apos
uma queda de sessao do Colab sem reprocessar nada. A escrita e feita num arquivo
`.parcial` e so entao renomeada — se a sessao cair no meio da gravacao, nao fica um
parquet truncado que o checkpoint trataria como pronto.

**Falhas.** Cada arquivo tem ate 3 tentativas, e a conexao e reaberta entre elas
(uma conexao FTP morta e a causa mais provavel de falha em sequencia). O que falhar
nas tres vai para `logs/pendentes.json`, que e regravado periodicamente durante o
laco, sem interromper o processo.

**Idade desconhecida.** Quando `COD_IDADE` vem fora do intervalo esperado,
`idade_anos` fica nulo e `faixa_etaria` tambem. Esse nulo e preservado como nulo
real no parquet (`astype('string')`, nunca `astype(str)`, que transformaria o
ausente na string literal `'nan'` — uma categoria a mais na agregacao, que jamais
casaria com a tabela de populacao no join). O total e somado ao longo de todos os
arquivos e reportado ao final.

**Duas protecoes descobertas testando esta celula contra o servidor real.** A
primeira: so erro de **rede** derruba e reabre a conexao. Um erro de processamento
(decode, parquet, disco) nao tem relacao com o FTP, e reconectar nesse caso produz
uma tempestade de reconexoes — em teste, 36 reconexoes em poucos segundos fizeram
o servidor passar a recusar novas conexoes por alguns minutos. A segunda: um teto
de 60 falhas aborta a execucao com mensagem clara, em vez de repetir o mesmo erro
1.944 vezes.


In [ ]:
import json
import os
import socket
import tempfile
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from ftplib import FTP, all_errors

import datasus_dbc
import pandas as pd
from dbfread import DBF

# 6 conexoes paralelas. Medido contra o servidor real: uma conexao unica sustenta
# 0,50 MB/s, seis sustentam 2,19-2,39 MB/s — cerca de 4,8x. Subir muito acima
# disso nao ajuda e aumenta a chance de o servidor recusar conexoes.
WORKERS = 6

# Limite de seguranca: se algo estiver sistematicamente errado (caminho invalido,
# disco cheio, biblioteca faltando), sem esse teto o laco tentaria e reconectaria
# 1.944 vezes, derrubando a propria conexao com o servidor.
MAX_PENDENTES = 60

TMP = tempfile.gettempdir()

# So erro de REDE justifica derrubar e reabrir a conexao. Erro de processamento
# (decode, parquet, disco) nao tem nada a ver com o FTP: reconectar nesse caso
# gera uma tempestade de reconexoes que o servidor passa a recusar.
ERROS_REDE = all_errors + (socket.error, EOFError, OSError)

COLUNAS = ['MUNIC_RES', 'SEXO', 'IDADE', 'COD_IDADE', 'DIAG_PRINC',
           'PROC_REA', 'IDENT', 'MORTE', 'VAL_TOT', 'DIAS_PERM']

SAIDA = ['cod_municipio_6', 'sexo', 'idade_anos', 'faixa_etaria', 'ano', 'mes',
         'amputacao', 'morte', 'val_tot', 'dias_perm']

trava = threading.Lock()
estado = {'baixados': 0, 'pulados': 0, 'bytes': 0, 'faixa_indefinida': 0}
pendentes = []
abortar = threading.Event()
inicio = time.time()


def conectar():
    con = FTP(HOST, timeout=300)
    con.login()
    con.cwd(DIR_FTP)
    return con


def gravar_pendentes():
    with open(f'{BASE}/logs/pendentes.json', 'w') as arquivo:
        json.dump(pendentes, arquivo, indent=2)


def registrar_falha(uf, ano, mes, categoria, erro):
    with trava:
        pendentes.append({'uf': uf, 'ano': ano, 'mes': mes,
                          'categoria': categoria, 'erro': str(erro)})
        print(f'FALHOU {uf} {ano}-{mes:02d} [{categoria}]: {erro}', flush=True)
        if len(pendentes) >= MAX_PENDENTES and not abortar.is_set():
            abortar.set()
            print(f'ABORTANDO: {len(pendentes)} falhas acumuladas. Algo esta '
                  f'sistematicamente errado — verifique logs/pendentes.json '
                  f'antes de reexecutar.', flush=True)


def processar(con, uf, ano, mes, wid):
    """Baixa, decodifica, filtra e grava um arquivo. Devolve (status, bytes, indefinidas)."""
    pasta = f'{BASE}/bronze/uf={uf}/ano={ano}'
    nome = f'RD{uf}{ano % 100:02d}{mes:02d}'
    destino = f'{pasta}/{nome}.parquet'
    if os.path.exists(destino):
        return 'pulado', 0, 0

    os.makedirs(pasta, exist_ok=True)
    dbc = os.path.join(TMP, f'w{wid}.dbc')
    dbf = os.path.join(TMP, f'w{wid}.dbf')

    with open(dbc, 'wb') as arquivo:
        con.retrbinary('RETR ' + nome + '.dbc', arquivo.write)
    tamanho = os.path.getsize(dbc)
    datasus_dbc.decompress(dbc, dbf)

    df = pd.DataFrame(iter(DBF(dbf, encoding='latin-1')))
    df = df[[c for c in COLUNAS if c in df.columns]]
    df = filtrar_internacoes_diabetes(df)
    df['idade_anos'] = idade_em_anos(df['IDADE'], df['COD_IDADE'])
    # astype('string') preserva idade desconhecida como nulo real; astype(str)
    # transformaria o ausente na string 'nan', que nunca casaria no join com a
    # tabela de populacao.
    df['faixa_etaria'] = faixa_etaria(df['idade_anos']).astype('string')
    indefinidas = int(df['faixa_etaria'].isna().sum())
    df = df.rename(columns={'MUNIC_RES': 'cod_municipio_6', 'SEXO': 'sexo',
                            'MORTE': 'morte', 'VAL_TOT': 'val_tot',
                            'DIAS_PERM': 'dias_perm'})
    df['ano'], df['mes'] = ano, mes

    # Grava num arquivo provisorio e so entao renomeia: se a sessao cair no meio
    # da escrita, nao sobra um parquet truncado que o checkpoint trataria como
    # pronto na proxima execucao.
    provisorio = destino + '.parcial'
    df[SAIDA].to_parquet(provisorio, index=False)
    os.replace(provisorio, destino)

    os.remove(dbc)
    os.remove(dbf)
    return 'baixado', tamanho, indefinidas


def relatar(total):
    decorrido = time.time() - inicio
    mb = estado['bytes'] / 1024 ** 2
    taxa = mb / max(decorrido, 1)
    restantes = len(alvos) - total
    eta = (restantes / max(total, 1)) * decorrido / 60
    print(f'[{total}/{len(alvos)}] {mb:.0f} MB, {taxa:.2f} MB/s, '
          f'{decorrido / 60:.1f} min decorridos, ~{eta:.0f} min restantes',
          flush=True)


def worker(wid, fatia):
    con = conectar()
    for uf, ano, mes in fatia:
        if abortar.is_set():
            break
        for tentativa in range(3):
            try:
                status, tamanho, indefinidas = processar(con, uf, ano, mes, wid)
                with trava:
                    if status == 'pulado':
                        estado['pulados'] += 1
                    else:
                        estado['baixados'] += 1
                        estado['bytes'] += tamanho
                        estado['faixa_indefinida'] += indefinidas
                    total = estado['baixados'] + estado['pulados']
                    if total % 100 == 0:
                        relatar(total)
                        gravar_pendentes()
                break
            except ERROS_REDE as erro:
                if tentativa == 2:
                    registrar_falha(uf, ano, mes, 'rede', erro)
                else:
                    try:
                        con.quit()
                    except Exception:
                        pass
                    time.sleep(3 * (tentativa + 1))
                    try:
                        con = conectar()
                    except Exception:
                        pass
            except Exception as erro:
                # Erro de processamento: a conexao esta boa, nao mexe nela.
                if tentativa == 2:
                    registrar_falha(uf, ano, mes, 'processamento', erro)
                else:
                    time.sleep(1)
    try:
        con.quit()
    except Exception:
        pass


fatias = [alvos[i::WORKERS] for i in range(WORKERS)]
with ThreadPoolExecutor(WORKERS) as executor:
    list(executor.map(lambda par: worker(par[0], par[1]), list(enumerate(fatias))))

gravar_pendentes()
decorrido = (time.time() - inicio) / 60
print(f'concluido em {decorrido:.1f} min')
print(f'  baixados agora: {estado["baixados"]}')
print(f'  ja existiam (pulados): {estado["pulados"]}')
print(f'  pendentes (falharam 3x): {len(pendentes)}')
print(f'  linhas com faixa etaria indefinida: {estado["faixa_indefinida"]}')
if abortar.is_set():
    print('ATENCAO: execucao abortada pelo limite de falhas. Corrija a causa e '
          'reexecute — o checkpoint preserva tudo que ja foi baixado.')


### 1.5 Verificação de completude

Confere quantos dos 1.944 parquets esperados foram de fato gerados. Tolera até 2%
de arquivos faltando (ex.: meses sem AIHs em UFs pequenas, indisponibilidade
pontual do FTP); acima disso, o `assert` interrompe a execução para que
`logs/pendentes.json` seja investigado antes de seguir para a camada silver.

In [ ]:
import glob
gerados = glob.glob(f'{BASE}/bronze/uf=*/ano=*/*.parquet')
print(f'gerados: {len(gerados)} de {len(alvos)}')
assert len(gerados) >= len(alvos) * 0.98, 'completude abaixo de 98% — investigar pendentes.json'

## Parte 2 — Camada silver (PySpark) e camada gold

A camada bronze tem uma linha por internação, espalhada em 1.944 arquivos parquet
pequenos. A camada silver agrega isso a nível de município/ano/sexo/faixa etária
usando PySpark — o volume total (dezenas de milhões de internações ao longo de
6 anos) torna essa agregação inviável em pandas puro dentro da memória do Colab.

In [ ]:
!pip install -q pyspark
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName('diabetes_sus')
         .config('spark.driver.memory', '8g')
         .config('spark.sql.shuffle.partitions', '64')
         .getOrCreate())
print(spark.version)

### 2.1 Agregação silver

Lê toda a árvore de partições da bronze, converte `morte` e `amputacao` para
inteiro (para poder somar) e agrega por `cod_municipio_6`, `ano`, `sexo` e
`faixa_etaria`, contando internações, somando amputações, óbitos, valor total
gasto e dias de permanência. O resultado é gravado em parquet (particionável e
eficiente para a próxima etapa) em `silver/municipio_ano_faixa_sexo`.

**Idade média (correção pós-revisão):** a bronze já grava `idade_anos` por
internação (Seção 1.4), só a agregação original descartava essa granularidade.
Para calcular idade média mais adiante sem aproximar por ponto médio de faixa
etária — o bucket `80+` é aberto, não tem limite superior natural, e usar um
valor arbitrário seria um número fabricado num trabalho avaliado — a
agregação agora também soma `idade_anos` em `idade_soma` e conta quantas
internações têm idade conhecida em `idade_validas`. Linhas com `idade_anos`
nulo (idade desconhecida — ver Seção 1.4) não entram nem na soma nem na
contagem: `F.sum` ignora nulo por padrão, e `F.count('idade_anos')` (ao
contrário de `F.count('*')`) só conta as linhas não nulas dessa coluna — o
denominador honesto da futura média.

**Se você já rodou a ingestão inteira antes desta correção:** não é
preciso baixar os `.dbc` de novo. A camada bronze já tem `idade_anos` desde
sempre — só a agregação silver não a resumia. Basta reexecutar a partir
desta célula (2.1) em diante: silver, junção e gold. A Parte 1 inteira
(Seções 1.1 a 1.5, o download e a completude da bronze) pode ser pulada.

In [ ]:
bronze = spark.read.parquet(f'{BASE}/bronze')

silver = (bronze
    .withColumn('morte', F.col('morte').cast('int'))
    .withColumn('amputacao', F.col('amputacao').cast('int'))
    .groupBy('cod_municipio_6', 'ano', 'sexo', 'faixa_etaria')
    .agg(
        F.count('*').alias('internacoes'),
        F.sum('amputacao').alias('amputacoes'),
        F.sum('morte').alias('obitos'),
        F.sum('val_tot').alias('val_total'),
        F.sum('dias_perm').alias('dias_perm_total'),
        # F.sum ignora nulo por padrao; F.count(coluna) so conta as linhas
        # NAO nulas dessa coluna (ao contrario de F.count('*')) -- por isso
        # internacoes com idade_anos desconhecida nao entram nem na soma
        # nem no denominador da futura idade media.
        F.sum('idade_anos').alias('idade_soma'),
        F.count('idade_anos').alias('idade_validas'),
    ))

silver.write.mode('overwrite').parquet(f'{BASE}/silver/municipio_ano_faixa_sexo')
print(silver.count(), 'linhas na silver')

### 2.2 Insumos do join

O join da silver com populacao e cobertura de APS depende de dois arquivos que
**nao sao gerados por este notebook**. Os dois ja estao versionados no
repositorio e vieram junto com o clone da Secao 1.1 — nao ha upload manual:

1. **`data/gold/populacao_municipio_faixa_sexo.parquet`** — populacao do Censo
   2022 por municipio, sexo e faixa etaria (5.570 municipios, 203.080.756
   habitantes), gerada por `scripts/baixar_populacao_ibge.py`.
2. **`data/gold/cobertura_aps.csv`** — cobertura de Atencao Primaria por
   municipio e ano, 2019-2024, gerada por `scripts/baixar_cobertura_aps.py` a
   partir da API publica do portal Relatorios Publicos da APS.

A celula abaixo confirma que os dois chegaram antes de seguir.


In [ ]:
import os

pop_path = f'{REPO}/data/gold/populacao_municipio_faixa_sexo.parquet'
aps_path = f'{REPO}/data/gold/cobertura_aps.csv'

faltando = [p for p in (pop_path, aps_path) if not os.path.exists(p)]
if faltando:
    raise FileNotFoundError(
        'Insumos ausentes no clone: ' + ', '.join(faltando) +
        '. Reexecute a celula 1.1 para refazer o clone do repositorio.'
    )
print('insumos encontrados no clone:')
for caminho in (pop_path, aps_path):
    print(f' - {caminho} ({os.path.getsize(caminho):,} bytes)')


### 2.3 Junção com população e cobertura de APS

Traz a silver para pandas (já agregada, portanto pequena o suficiente), expande o
código de município do SIH (6 dígitos, sem dígito verificador) para o código IBGE
de 7 dígitos usando a população como referência (`mapa_6_para_7` /
`completar_codigo`), e então junta com população e cobertura de APS.

**A gold é construída a partir da grade completa, não da silver (correção C1 da
revisão final).** A camada bronze só contém internações por diabetes, então a
silver só tem linha para `(município, ano, sexo, faixa etária)` que registrou ao
menos uma internação. Se a gold nascesse dessas linhas, a população das
combinações **sem** internação nunca entraria no denominador — e a padronização
etária, que existe justamente para tornar municípios comparáveis, passaria a
inflar sistematicamente a taxa dos municípios pequenos (que têm mais faixas
vazias). A faixa `<30` sozinha vale cerca de 42% da população padrão e é a que
menos interna por diabetes.

Por isso a célula monta primeiro a **grade completa** — o produto cartesiano da
tabela de população (5.570 municípios × 2 sexos × 7 faixas) pelos 6 anos, ou seja
**467.880 linhas** — e faz `left join` da silver sobre ela, preenchendo com zero
`internacoes`, `amputacoes`, `obitos`, `val_total`, `dias_perm_total`,
`idade_soma` e `idade_validas` onde não houve internação. Zero internações em
uma faixa passa a ser um fato registrado, não uma linha ausente.

Três verificações importantes:

- **Órfãos:** linhas cujo código de 6 dígitos não teve correspondência de 7
  dígitos na tabela de população viram `NaN` em `cod_municipio` e são
  descartadas. O `assert` interrompe a execução se mais de 1% das linhas caírem
  nessa situação — um volume alto de órfãos indicaria problema na tabela de
  população ou no mapeamento, não apenas ruído esperado.
- **Código de sexo:** o SIH usa `1` para masculino e `3` para feminino. Qualquer
  outro valor é tratado como desconhecido: vira `NaN` em `sexo`. A célula imprime
  quantas linhas caíram nessa situação.
- **Internações que a grade não posiciona:** com a grade completa, uma internação
  sem município válido, sem sexo válido ou sem faixa etária (idade desconhecida —
  ver Seção 1.4) não tem célula onde ser somada e fica de fora da gold. O total
  dessas internações é impresso, para que a perda seja um número conhecido e não
  um silêncio.

Ao final, a UF e a macrorregião são derivadas do próprio código de município
(`uf_do_codigo`, `regiao_da_uf`), as colunas são selecionadas explicitamente na
ordem do esquema e o resultado é gravado como `gold/municipio_ano.csv`, com uma
linha por município/ano/sexo/faixa etária e exatamente **15 colunas**:

`cod_municipio`, `uf`, `regiao`, `ano`, `sexo`, `faixa_etaria`, `internacoes`,
`amputacoes`, `obitos`, `val_total`, `dias_perm_total`, `populacao`,
`cobertura_aps`, `idade_soma`, `idade_validas`.

A seleção explícita (`gold = gold[COLUNAS_GOLD]`) existe para que o esquema
gravado seja o esquema documentado — antes da revisão final a coluna auxiliar
`cod_municipio_6` sobrevivia até o `to_csv` e a saída tinha 16 colunas contra as
15 listadas aqui. Com a grade, `cod_municipio_6` nem chega a existir no quadro
final, e a seleção mantém a garantia caso a célula mude.

**`idade_soma` e `idade_validas`:** vêm prontas da silver (Seção 2.1) e
atravessam esta célula sem transformação — o `merge` não as toca, só as
posiciona na grade e preenche com zero onde não houve internação. Elas existem
para permitir calcular idade média por qualquer agrupamento mais adiante
(`idade_soma.sum() / idade_validas.sum()`, por exemplo por região e sexo em
`03_indice_icvd.ipynb`) sem aproximar por ponto médio de faixa etária.

In [ ]:
import os
import pandas as pd
from diabetes_sus.municipios import completar_codigo, mapa_6_para_7, regiao_da_uf, uf_do_codigo

# Idempotente: garante o diretorio tambem para quem retomou a execucao a partir
# da Parte 2 e nao rodou a celula 1.1 nesta sessao.
os.makedirs(f'{BASE}/gold', exist_ok=True)

pop = pd.read_parquet(f'{REPO}/data/gold/populacao_municipio_faixa_sexo.parquet')
# So as tres colunas usadas no join: o CSV tambem traz uf, regiao e fonte, que
# colidiriam ou entrariam de carona na gold sem necessidade.
aps = pd.read_csv(f'{REPO}/data/gold/cobertura_aps.csv',
                  dtype={'cod_municipio': str},
                  usecols=['cod_municipio', 'ano', 'cobertura_aps'])

s = spark.read.parquet(f'{BASE}/silver/municipio_ano_faixa_sexo').toPandas()

mapa = mapa_6_para_7(pop['cod_municipio'].unique())
s['cod_municipio'] = completar_codigo(s['cod_municipio_6'], mapa)

orfaos = s['cod_municipio'].isna().sum()
print(f'linhas sem municipio correspondente: {orfaos}')
assert orfaos / len(s) < 0.01, 'mais de 1% de orfaos - investigar antes de seguir'

sexo_bruto = s['sexo']
s['sexo'] = sexo_bruto.map({'1': 'M', '3': 'F', 1: 'M', 3: 'F'})
sexo_nao_mapeado = (sexo_bruto.notna() & s['sexo'].isna()).sum()
print(f'linhas com sexo fora de 1/3 (tratadas como desconhecidas): {sexo_nao_mapeado}')

CHAVE = ['cod_municipio', 'ano', 'sexo', 'faixa_etaria']
CONTAGENS = ['internacoes', 'amputacoes', 'obitos', 'val_total',
             'dias_perm_total', 'idade_soma', 'idade_validas']

# GRADE COMPLETA (correcao C1): todas as combinacoes municipio x ano x sexo x
# faixa da tabela de populacao -- 5.570 x 6 x 2 x 7 = 467.880 linhas. A silver
# so tem linha onde houve internacao; sem a grade, a populacao das combinacoes
# sem caso nunca entraria no denominador e a padronizacao etaria inflaria a taxa
# dos municipios pequenos. Ver docs/03-modelagem.md, Secao 3.3.
anos = pd.DataFrame({'ano': list(range(2019, 2025))})
grade = pop.drop(columns=['uf', 'regiao'], errors='ignore').merge(anos, how='cross')
print(f'grade completa: {len(grade)} linhas (esperado 467880)')

agregada = (s.dropna(subset=['cod_municipio', 'sexo', 'faixa_etaria'])
             .groupby(CHAVE, as_index=False, observed=True)[CONTAGENS].sum())

# Internacao sem municipio, sexo ou faixa validos nao tem celula na grade onde
# ser somada. O volume fica visivel aqui em vez de sumir em silencio.
fora_da_grade = int(s['internacoes'].sum() - agregada['internacoes'].sum())
print(f'internacoes sem municipio/sexo/faixa validos, fora da gold: {fora_da_grade}')

gold = (grade.merge(agregada, on=CHAVE, how='left')
             .merge(aps, on=['cod_municipio', 'ano'], how='left'))
# Zero internacoes na combinacao passa a ser um fato registrado, nao uma linha
# ausente -- e' isso que devolve o peso da faixa vazia ao denominador.
gold[CONTAGENS] = gold[CONTAGENS].fillna(0)
gold['uf'] = uf_do_codigo(gold['cod_municipio'])
gold['regiao'] = regiao_da_uf(gold['uf'])

COLUNAS_GOLD = ['cod_municipio', 'uf', 'regiao', 'ano', 'sexo', 'faixa_etaria',
                'internacoes', 'amputacoes', 'obitos', 'val_total',
                'dias_perm_total', 'populacao', 'cobertura_aps',
                'idade_soma', 'idade_validas']
gold = gold[COLUNAS_GOLD]

gold.to_csv(f'{BASE}/gold/municipio_ano.csv', index=False)
print(gold.shape)   # esperado (467880, 15)

### 2.4 Etapas finais (fora do Colab)

Estas etapas não são células de código porque acontecem fora do ambiente do
Colab, na máquina local:

1. **Baixar o resultado:** copie `{BASE}/gold/municipio_ano.csv` do Google Drive
   para `data/gold/municipio_ano.csv` no repositório local.
2. **Verificar localmente**, no terminal do repositório:

   ```bash
   python -c "import pandas as pd; d=pd.read_csv('data/gold/municipio_ano.csv', dtype={'cod_municipio':str}); print(d.shape); print(d['ano'].value_counts().sort_index()); print(d.isna().sum())"
   ```

   O esperado é `(467880, 15)`, os seis anos de 2019 a 2024 com 77.980 linhas
   cada, e nenhum nulo em `internacoes`, `populacao` e `cod_municipio` — com a
   grade completa, essas três colunas são preenchidas por construção. Nulos em
   `cobertura_aps` continuam aceitáveis caso o e-Gestor não tenha dado para
   algum município/ano.
3. **Versionar:** `git add notebooks/01_ingestao_colab.ipynb data/gold/municipio_ano.csv`
   e commit — nunca versionar `.dbc`, `.dbf`, `data/bronze/` ou `data/silver/`
   (já cobertos pelo `.gitignore` do projeto).